# CF Farmer - Jupyter Notebook

Cloudflare Workers AI account farmer.
Auto-creates CF accounts, solves Turnstile, verifies email, generates API tokens.

## Step 1: Install Dependencies

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "curl_cffi>=0.5", "python-dotenv>=1.0", "requests"], check=True)
print("Done")

## Step 2: Config

In [ ]:
import os

# === EDIT THIS ===
os.environ["IMAP_HOST"] = "imap.gmail.com"
os.environ["IMAP_PORT"] = "993"
os.environ["IMAP_USER"] = "wllmstevan@gmail.com"     # Your Gmail
os.environ["IMAP_PASS"] = "your-app-password"          # Gmail app password
os.environ["DOMAIN"] = "airwallex.fun"                 # Email domain

# Boterdrop solver (local) or Solverify (cloud)
USE_LOCAL_SOLVER = True  # False = use Solverify
LOCAL_SOLVER_URL = "http://localhost:8000"  # Boterdrop server
SOLVERIFY_API_KEY = ""  # Only if USE_LOCAL_SOLVER = False

print(f"Domain: {os.environ['DOMAIN']}")
print(f"Solver: {'Local (Boterdrop)' if USE_LOCAL_SOLVER else 'Solverify'}")

## Step 3: Add Proxies

In [ ]:
# Add proxies (one per line)
# Format: host:port:user:pass
proxies = """
# Add your proxies here
# gw.dataimpulse.com:823:997616a254f42231d6bc:2b18a1af1ffcf881
""".strip()

with open("proxy.txt", "w") as f:
    f.write(proxies)

print(f"Proxies saved: {len(proxies.splitlines())} lines")

## Step 4: Patch farmer.py for Local Solver

In [ ]:
# If using local Boterdrop solver, patch farmer.py
if USE_LOCAL_SOLVER:
    import re
    
    with open("farmer.py", "r") as f:
        code = f.read()
    
    # Replace Solverify solve_turnstile with local solver call
    local_solver_func = '''
async def solve_turnstile(captcha_key, sitekey, log=print):
    """Solve Turnstile via local Boterdrop solver."""
    import requests
    url = f"{os.environ.get('LOCAL_SOLVER_URL', 'http://localhost:8000')}/turnstile"
    log(f"    [captcha] solving via local solver...")
    for attempt in range(3):
        try:
            r = requests.get(url, params={"sitekey": sitekey, "url": SIGNUP_URL}, timeout=120)
            data = r.json()
            if data.get("token"):
                log(f"    [captcha] solved!")
                return data["token"]
            log(f"    [captcha] attempt {attempt+1} failed: {data}")
        except Exception as e:
            log(f"    [captcha] attempt {attempt+1} error: {e}")
    return None
'''
    
    # Replace the Solverify function
    pattern = r'# --- Solverify Turnstile ---.*?return None\n'
    code = re.sub(pattern, local_solver_func + '\n', code, flags=re.DOTALL)
    
    with open("farmer.py", "w") as f:
        f.write(code)
    
    print("farmer.py patched for local solver")
else:
    print("Using Solverify, no patch needed")

## Step 5: Start Boterdrop Solver (if using local)

In [ ]:
import subprocess, time, requests

solver_proc = None
if USE_LOCAL_SOLVER:
    # Check if Boterdrop is already running
    try:
        r = requests.get(f"{LOCAL_SOLVER_URL}/", timeout=2)
        print(f"Boterdrop already running at {LOCAL_SOLVER_URL}")
    except:
        print("Starting Boterdrop solver...")
        solver_proc = subprocess.Popen(
            [sys.executable, "api_server.py"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT
        )
        # Wait for ready
        for i in range(30):
            time.sleep(2)
            try:
                r = requests.get(f"{LOCAL_SOLVER_URL}/", timeout=2)
                if r.status_code == 200:
                    print(f"Solver ready after {(i+1)*2}s")
                    break
            except:
                pass
        else:
            print("Solver may not be ready, check logs")
else:
    print("Using Solverify, no local solver needed")

## Step 6: Run Farmer (1 Account)

In [ ]:
import asyncio

# Set solver env var for farmer
if USE_LOCAL_SOLVER:
    os.environ["LOCAL_SOLVER_URL"] = LOCAL_SOLVER_URL

# Import farmer
from farmer import farm_one

# Farm 1 account
async def run():
    captcha_key = SOLVERIFY_API_KEY if not USE_LOCAL_SOLVER else "local"
    acc = await farm_one(captcha_key, log=print)
    if acc:
        print(f"\n=== SUCCESS ===")
        print(f"Email: {acc['email']}")
        print(f"Account ID: {acc['account_id']}")
        print(f"API Token: {acc['api_token'][:30]}...")
        print(f"Status: {acc['status']}")
    else:
        print("\n=== FAILED ===")
    return acc

acc = await run()

## Step 7: Run Batch (N Accounts)

In [ ]:
from farmer import farm_batch

# Farm N accounts
COUNT = 5  # How many accounts to create

captcha_key = SOLVERIFY_API_KEY if not USE_LOCAL_SOLVER else "local"
await farm_batch(captcha_key, COUNT)

## Step 8: Check Results

In [ ]:
import json

# Load created accounts
if os.path.exists("accounts.json"):
    with open("accounts.json") as f:
        accounts = json.load(f)
    
    print(f"Total accounts: {len(accounts)}")
    print()
    for acc in accounts:
        print(f"  {acc['email']} | {acc['status']} | {acc.get('account_id', 'N/A')}")
else:
    print("No accounts yet")

## Step 9: Verify 9router Integration

In [ ]:
import sqlite3

# Check 9router DB for injected connections
db_path = os.path.expanduser("~/.9router/db/data.sqlite")
if os.path.exists(db_path):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("SELECT name, isActive FROM providerConnections WHERE provider='cloudflare-ai'")
    rows = cur.fetchall()
    conn.close()
    
    print(f"Cloudflare connections in 9router: {len(rows)}")
    for name, active in rows:
        print(f"  {name} - {'active' if active else 'inactive'}")
else:
    print("9router DB not found")

## Step 10: Cleanup

In [ ]:
# Stop solver if we started it
if solver_proc:
    solver_proc.terminate()
    solver_proc.wait()
    print("Solver stopped")
else:
    print("No solver to stop")